# Databricks Notebook 2: Performance Optimization & Advanced Lakehouse Techniques
**Target Audience:** Azure Data Engineers & Senior Technical Interview Preparation  
**Objective:** Master hands-on PySpark optimization, storage format benchmarks, Delta Lake internals, and advanced data engineering patterns on **high-volume, realistic datasets**.

---

### Complete Curriculum (One Comprehensive Task Per Cell):
1. **Task 0: Interactive Setup & High-Volume Data Scaling Configuration**
2. **Task 1: Partitioning (`partitionBy`) & Partition Pruning (150,000+ Rows)**
3. **Task 2: Broadcast Join Optimization (`broadcast()`) vs Shuffle Joins (500,000+ Fact Rows)**
4. **Task 3: Storage Format Size Benchmark (CSV vs Parquet vs Delta on 100,000+ Rows)**
5. **Task 4: Caching vs Persistence (`cache()`, `persist()`, StorageLevels on 1,000,000 Rows)**
6. **Task 5: Delta Lake File Lifecycle & `VACUUM` Internals (50,000 Rows Bulk Lifecycle)**
7. **Task 6: Delta Lake Liquid Clustering (`CLUSTER BY`) vs Traditional Partitioning (150,000 Rows)**
8. **Task 7: Views Architecture: Local Temp View vs Global Temp View vs Semantic View (50,000 Rows)**
9. **Task 8: Delta Live Tables (DLT) Declarative Medallion Pipelines & Data Quality Expectations**
10. **Task 9: Slowly Changing Dimensions (SCD Type 1 & SCD Type 2) in Pure PySpark (50,000 Base + 10,000 CDC)**
11. **Task 10: Change Data Capture (CDC) & Delta Change Data Feed (CDF) Internals (20,000+ Records)**
12. **Task 11: Optional Practice Environment Cleanup**

In [ ]:
# Task 0: Interactive Setup & High-Volume Data Configuration
# Configures widgets, storage paths (ADLS Gen2 with DBFS fallback), and data volume scaling.

import time
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, current_timestamp, current_date, expr, rand, when,
    broadcast, concat_ws, date_add, to_date, row_number, sha2, count, sum, avg, round
)
from pyspark.sql.types import (
    StructType, StructField, IntegerType, StringType, DoubleType, DateType, TimestampType, BooleanType
)
from pyspark.sql.window import Window
from pyspark.storagelevel import StorageLevel
from delta.tables import DeltaTable

# Widgets for interactive configuration
dbutils.widgets.text("storage_account", "azureprojectdevadls01", "ADLS Gen2 Storage Account")
dbutils.widgets.text("container", "raw", "Container Name")
dbutils.widgets.text("base_folder", "practice_optimization_large", "Base Folder in Container")
dbutils.widgets.text("target_catalog", "bronze_catalog", "Target Unity Catalog")
dbutils.widgets.text("target_schema", "practice", "Target Schema")
dbutils.widgets.dropdown("data_scale", "HIGH", ["MEDIUM", "HIGH", "STRESS_TEST"], "Data Scale Mode")

storage_account = dbutils.widgets.get("storage_account").strip()
container       = dbutils.widgets.get("container").strip()
base_folder     = dbutils.widgets.get("base_folder").strip()
target_catalog  = dbutils.widgets.get("target_catalog").strip()
target_schema   = dbutils.widgets.get("target_schema").strip()
data_scale      = dbutils.widgets.get("data_scale").strip()

# Dynamic scale factor:
# MEDIUM: ~50,000 - 100,000 rows (fastest execution)
# HIGH (Default): ~150,000 - 500,000 rows (optimal realistic interview practice)
# STRESS_TEST: ~1,000,000 - 2,000,000 rows (cluster heavy performance test)
scale_multiplier = 1.0 if data_scale == "HIGH" else (0.3 if data_scale == "MEDIUM" else 2.5)

# Primary ADLS Gen2 path & DBFS fallback
adls_base_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net/{base_folder}"
dbfs_base_path = f"/tmp/{base_folder}"

try:
    dbutils.fs.ls(f"abfss://{container}@{storage_account}.dfs.core.windows.net/")
    active_base_path = adls_base_path
    print(f"[Connected] Using ADLS Gen2 Storage: {active_base_path}")
except Exception as e_storage:
    active_base_path = dbfs_base_path
    print(f"[Fallback Active] Using Workspace DBFS Storage: {active_base_path}")
    print(f"Reason: External storage credentials not mounted ({str(e_storage)[:80]}...)")

print(f"Scale Mode: {data_scale} (Multiplier: {scale_multiplier}x)")
print(f"Target Catalog.Schema: {target_catalog}.{target_schema}")

In [ ]:
# Task 1: Partitioning - partitionBy & Partition Pruning (150,000+ Rows)
# Concept: Physical directory organization based on column values.
# Benefit: Query engines skip irrelevant partition folders (Partition Pruning).
# Interview Watchout: Avoid high-cardinality columns (e.g. Timestamp, UserID) -> Small Files Anti-Pattern!

partition_rows = int(150000 * scale_multiplier)
partition_demo_path = f"{active_base_path}/partition_demo"

print(f"Generating {partition_rows:,} realistic sales transaction records across multiple Years & Regions...")

df_sales = spark.range(1, partition_rows + 1) \
    .withColumnRenamed("id", "TransactionID") \
    .withColumn("CustomerID", (col("TransactionID") % 5000) + 1) \
    .withColumn("Year", when(col("TransactionID") % 4 == 0, 2023)
                        .when(col("TransactionID") % 4 == 1, 2024)
                        .when(col("TransactionID") % 4 == 2, 2025)
                        .otherwise(2026)) \
    .withColumn("Region", when(col("TransactionID") % 4 == 0, "North")
                          .when(col("TransactionID") % 4 == 1, "South")
                          .when(col("TransactionID") % 4 == 2, "East")
                          .otherwise("West")) \
    .withColumn("ProductCategory", when(col("TransactionID") % 5 == 0, "Auto Insurance")
                                  .when(col("TransactionID") % 5 == 1, "Property Insurance")
                                  .when(col("TransactionID") % 5 == 2, "Life Insurance")
                                  .when(col("TransactionID") % 5 == 3, "Health Insurance")
                                  .otherwise("Commercial Liability")) \
    .withColumn("Amount", (rand() * 2500 + 50).cast("decimal(10,2)")) \
    .withColumn("TransactionDate", date_add(to_date(lit("2023-01-01")), (col("TransactionID") % 1200).cast("int")))

print("Sample Sales Data:")
df_sales.show(5, truncate=False)

# Write to storage partitioned by Year and Region
print(f"Writing partitioned Delta table to: {partition_demo_path}...")
df_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("Year", "Region") \
    .save(partition_demo_path)

# Inspect physical directory layout
print("\n--- Physical Storage Directory Layout (Partition Folders) ---")
for year_folder in dbutils.fs.ls(partition_demo_path):
    if "Year=" in year_folder.name:
        print(f"  [Year Directory]: {year_folder.name}")
        for region_folder in dbutils.fs.ls(year_folder.path):
            if "Region=" in region_folder.name:
                print(f"      └── [Region Directory]: {region_folder.name}")

# Measure Partition Pruning Performance
print("\n--- Benchmark: Query WITH Partition Pruning vs WITHOUT ---")
t0 = time.time()
count_pruned = spark.read.format("delta").load(partition_demo_path) \
    .filter((col("Year") == 2025) & (col("Region") == "North")).count()
t_pruned = time.time() - t0

t0 = time.time()
count_unpruned = spark.read.format("delta").load(partition_demo_path) \
    .filter(col("ProductCategory") == "Life Insurance").count()
t_unpruned = time.time() - t0

print(f"Query WITH Partition Pruning (Year=2025, Region=North) : {t_pruned:.3f}s (Scanned ONLY 1 folder!)")
print(f"Query WITHOUT Partition Pruning (Filter on non-partition col): {t_unpruned:.3f}s (Full Table Scan across all 16 folders!)")

# Physical Plan
print("\n--- Physical Plan Showing PartitionFilters ---")
spark.read.format("delta").load(partition_demo_path) \
    .filter((col("Year") == 2025) & (col("Region") == "North")).explain(True)

In [ ]:
# Task 2: Broadcast Join Optimization (500,000 Fact Rows vs Dimension)
# Concept: Joining a 500,000 row fact table with a small 10-row dimension table.
# Standard Join: Shuffles 500,000 rows across executor nodes (network I/O, disk spill).
# Broadcast Join: Copies the tiny 10-row dimension to all executors, completely avoiding shuffle!

fact_rows = int(500000 * scale_multiplier)
print(f"Generating {fact_rows:,} high-volume Fact transactions...")

df_fact = spark.range(1, fact_rows + 1) \
    .withColumnRenamed("id", "TransactionID") \
    .withColumn("StoreID", (col("TransactionID") % 10) + 1) \
    .withColumn("CustomerID", (col("TransactionID") % 25000) + 1) \
    .withColumn("SaleAmount", (rand() * 850 + 25).cast("decimal(10,2)")) \
    .withColumn("PaymentType", when(col("TransactionID") % 3 == 0, "CREDIT_CARD")
                              .when(col("TransactionID") % 3 == 1, "DEBIT_CARD")
                              .otherwise("DIGITAL_WALLET"))

# Small Dimension Table (10 Stores, ~few KB)
stores_data = [
    (1, "Downtown Store", "New York", "NY", 0.08875),
    (2, "Suburbs Plaza", "Chicago", "IL", 0.1025),
    (3, "Metro Hub", "Dallas", "TX", 0.0825),
    (4, "Tech Park", "San Francisco", "CA", 0.08625),
    (5, "Harbor View", "Seattle", "WA", 0.101),
    (6, "Central Market", "Boston", "MA", 0.0625),
    (7, "Airport Galleria", "Atlanta", "GA", 0.089),
    (8, "Valley Mall", "Phoenix", "AZ", 0.086),
    (9, "Lakeside Center", "Minneapolis", "MN", 0.080),
    (10, "Mile High Outlet", "Denver", "CO", 0.0881)
]
df_stores = spark.createDataFrame(stores_data, ["StoreID", "StoreName", "City", "State", "TaxRate"])

# Benchmark 1: Force Non-Broadcast Join (SortMergeJoin with Shuffle)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
t0 = time.time()
df_non_broadcast = df_fact.join(df_stores, on="StoreID", how="inner") \
    .withColumn("TotalWithTax", (col("SaleAmount") * (lit(1) + col("TaxRate"))).cast("decimal(10,2)"))
cnt_nb = df_non_broadcast.count()
t_non_broadcast = time.time() - t0

print(f"Non-Broadcast SortMergeJoin (with network shuffle) : {t_non_broadcast:.3f} seconds ({cnt_nb:,} rows)")

# Benchmark 2: Explicit Broadcast Hash Join (No Shuffle)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10485760)
t0 = time.time()
df_broadcast = df_fact.join(broadcast(df_stores), on="StoreID", how="inner") \
    .withColumn("TotalWithTax", (col("SaleAmount") * (lit(1) + col("TaxRate"))).cast("decimal(10,2)"))
cnt_b = df_broadcast.count()
t_broadcast = time.time() - t0

print(f"Optimized BroadcastHashJoin (Zero network shuffle) : {t_broadcast:.3f} seconds (Speedup: {t_non_broadcast / max(t_broadcast, 0.001):.1f}x faster!)")

print("\n--- Physical Plan Comparison ---")
print("Non-Broadcast Join Plan:")
df_non_broadcast.explain()
print("\nBroadcast Join Plan:")
df_broadcast.explain()

In [ ]:
# Task 3: Storage Format Size Benchmark on 100,000+ Enterprise Rows
# Concept: Parquet/Delta use Columnar Storage, Dictionary Encoding, and Snappy Compression.
# CSV is row-based uncompressed text. We generate 100,000 realistic records and compare file sizes in ADLS.

benchmark_rows = int(100000 * scale_multiplier)
benchmark_base = f"{active_base_path}/benchmark_size_large"
csv_path       = f"{benchmark_base}/data_csv"
parquet_path   = f"{benchmark_base}/data_parquet"
delta_path     = f"{benchmark_base}/data_delta"

print(f"Generating {benchmark_rows:,} rich enterprise insurance policy records...")

df_benchmark = spark.range(1, benchmark_rows + 1) \
    .withColumnRenamed("id", "PolicyID") \
    .withColumn("PolicyNumber", concat_ws("-", lit("POL"), (col("PolicyID") + 1000000).cast("string"))) \
    .withColumn("CustomerName", concat_ws(" ", lit("Customer_First_Middle_Last_Name_"), (col("PolicyID") % 2500).cast("string"))) \
    .withColumn("PolicyType", when(col("PolicyID") % 4 == 0, "Comprehensive Automobile Protection Guarantee")
                             .when(col("PolicyID") % 4 == 1, "Homeowners Standard Structural & Content Comprehensive")
                             .when(col("PolicyID") % 4 == 2, "Commercial Enterprise General Umbrella Liability")
                             .otherwise("Individual Universal Life & Disability Protection Plan")) \
    .withColumn("State", when(col("PolicyID") % 5 == 0, "California")
                         .when(col("PolicyID") % 5 == 1, "Texas")
                         .when(col("PolicyID") % 5 == 2, "Florida")
                         .when(col("PolicyID") % 5 == 3, "New York")
                         .otherwise("Illinois")) \
    .withColumn("City", when(col("PolicyID") % 5 == 0, "Los Angeles")
                        .when(col("PolicyID") % 5 == 1, "Houston")
                        .when(col("PolicyID") % 5 == 2, "Miami")
                        .when(col("PolicyID") % 5 == 3, "New York City")
                        .otherwise("Chicago")) \
    .withColumn("AnnualPremium", (rand() * 4500 + 400).cast("decimal(10,2)")) \
    .withColumn("Deductible", (when(col("PolicyID") % 3 == 0, 500).when(col("PolicyID") % 3 == 1, 1000).otherwise(2500)).cast("decimal(10,2)")) \
    .withColumn("CoverageLimit", (rand() * 1000000 + 100000).cast("decimal(12,2)")) \
    .withColumn("IsActive", when(col("PolicyID") % 7 == 0, False).otherwise(True)) \
    .withColumn("EffectiveDate", date_add(to_date(lit("2022-01-01")), (col("PolicyID") % 1400).cast("int"))) \
    .withColumn("ExpirationDate", date_add(col("EffectiveDate"), 365)) \
    .withColumn("UnderwritingAuditNote", concat_ws(" | ", lit("VERIFIED_AUDIT_RISK_TIER_A"), sha2(col("PolicyID").cast("string"), 256)))

# Write in all 3 formats
print(f"1. Writing {benchmark_rows:,} rows to CSV format...")
df_benchmark.write.mode("overwrite").option("header", "true").csv(csv_path)

print(f"2. Writing {benchmark_rows:,} rows to Parquet format (Snappy compressed)...")
df_benchmark.write.mode("overwrite").parquet(parquet_path)

print(f"3. Writing {benchmark_rows:,} rows to Delta Lake format...")
df_benchmark.write.mode("overwrite").format("delta").save(delta_path)

# Calculate directory sizes
def get_dir_size_bytes(dir_path):
    total_bytes = 0
    try:
        for f in dbutils.fs.ls(dir_path):
            if f.isDir():
                total_bytes += get_dir_size_bytes(f.path)
            else:
                total_bytes += f.size
    except Exception:
        pass
    return total_bytes

csv_bytes     = get_dir_size_bytes(csv_path)
parquet_bytes = get_dir_size_bytes(parquet_path)
delta_bytes   = get_dir_size_bytes(delta_path)

# Measure read scan speed
t0 = time.time()
csv_cnt = spark.read.option("header", "true").csv(csv_path).filter(col("AnnualPremium") > 2000).count()
t_csv_read = time.time() - t0

t0 = time.time()
parquet_cnt = spark.read.parquet(parquet_path).filter(col("AnnualPremium") > 2000).count()
t_parquet_read = time.time() - t0

# Print Report
print("\n" + "="*85)
print(f"  STORAGE BENCHMARK REPORT: CSV VS PARQUET VS DELTA ({benchmark_rows:,} ROWS)")
print("="*85)
print(f"{ 'Format':<15} | { 'Total Size (Bytes)':<20} | { 'Size (MB)':<12} | { 'Compression vs CSV':<22}")
print("-"*85)
print(f"{ 'CSV (Plain Text)':<15} | { csv_bytes:<20,} | { (csv_bytes/(1024*1024)):<12.2f} | { '0.00% (Baseline)':<22}")
print(f"{ 'Parquet (Snappy)':<15} | { parquet_bytes:<20,} | { (parquet_bytes/(1024*1024)):<12.2f} | { f'{(1 - parquet_bytes/csv_bytes)*100:.2f}% Smaller':<22}")
print(f"{ 'Delta Lake':<15} | { delta_bytes:<20,} | { (delta_bytes/(1024*1024)):<12.2f} | { f'{(1 - delta_bytes/csv_bytes)*100:.2f}% Smaller':<22}")
print("="*85)
print(f"Read Filter Benchmark (Filter on AnnualPremium):")
print(f"  • CSV Scan Time     : {t_csv_read:.3f} seconds (Reads entire text row by row)")
print(f"  • Parquet Scan Time : {t_parquet_read:.3f} seconds (Projection pruning & dictionary scan - {t_csv_read/max(t_parquet_read,0.001):.1f}x faster!)")

In [ ]:
# Task 4: Caching & Persistence Benchmark on 1,000,000 Rows
# Concept:
# - cache(): Default shortcut for persist(StorageLevel.MEMORY_AND_DISK_DESER).
# - persist(StorageLevel): Explicit control over where data lives (Memory, Disk, Off-Heap, Serialized).
# - Crucial Rule: Always unpersist() cached DataFrames when finished to avoid Executor OOM!

heavy_rows = int(1000000 * scale_multiplier)
print(f"Generating computationally heavy dataset with {heavy_rows:,} rows (SHA-256 rounds + trigonometric math)..."
)

df_heavy = spark.range(1, heavy_rows + 1) \
    .withColumn("HashLevel1", sha2(col("id").cast("string"), 256)) \
    .withColumn("HashLevel2", sha2(col("HashLevel1"), 256)) \
    .withColumn("TrigMetric", expr("sin(id) * cos(id) + sqrt(id)"))

# 1. Without Cache: Spark re-evaluates the entire execution DAG from scratch on every action!
print("--- Action Execution Times WITHOUT Cache ---")
t0 = time.time()
c1 = df_heavy.filter(col("TrigMetric") > 200).count()
t_uncached_1 = time.time() - t0
print(f"Action 1 (No Cache) : {t_uncached_1:.3f}s (Result: {c1:,} rows)")

t0 = time.time()
c2 = df_heavy.filter(col("TrigMetric") > 500).count()
t_uncached_2 = time.time() - t0
print(f"Action 2 (No Cache) : {t_uncached_2:.3f}s (Result: {c2:,} rows - Recomputed entire DAG!)")

# 2. Apply persist(StorageLevel.MEMORY_AND_DISK)
df_heavy.persist(StorageLevel.MEMORY_AND_DISK)

# Materialize cache with initial action
t0 = time.time()
df_heavy.count()
t_materialize = time.time() - t0
print(f"\nCache Materialization Run : {t_materialize:.3f}s (Data stored in RAM / spill to disk)")

# 3. With Cache: Subsequent actions read directly from memory
print("--- Action Execution Times WITH Cache ---")
t0 = time.time()
c3 = df_heavy.filter(col("TrigMetric") > 200).count()
t_cached_1 = time.time() - t0
print(f"Action 1 (With Cache) : {t_cached_1:.3f}s (Speedup: {t_uncached_1 / max(t_cached_1, 0.001):.1f}x faster!)")

t0 = time.time()
c4 = df_heavy.filter(col("TrigMetric") > 500).count()
t_cached_2 = time.time() - t0
print(f"Action 2 (With Cache) : {t_cached_2:.3f}s")
print(f"Is DataFrame Cached? -> {df_heavy.is_cached}")

# Always unpersist when finished
df_heavy.unpersist()
print(f"After unpersist(): Is DataFrame Cached? -> {df_heavy.is_cached} (RAM reclaimed)")

In [ ]:
# Task 5: How VACUUM Works in Delta Lake (50,000 Rows Bulk Lifecycle)
# Concept:
# Delta Lake uses Multi-Version Concurrency Control (MVCC). Updates/Deletes never delete files in-place;
# they write NEW Parquet files and flag older files as tombstoned in the _delta_log.
# VACUUM physically deletes files that are no longer referenced and are older than a retention threshold.

vacuum_rows = int(50000 * scale_multiplier)
vacuum_demo_path = f"{active_base_path}/vacuum_large_demo"

print(f"1. Ingesting initial batch of {vacuum_rows:,} customer policy records (Version 0)...")
spark.range(1, vacuum_rows + 1) \
    .withColumnRenamed("id", "PolicyID") \
    .withColumn("Status", when(col("PolicyID") % 5 == 0, "PENDING").otherwise("ACTIVE")) \
    .withColumn("RiskScore", (rand() * 100).cast("int")) \
    .write.format("delta").mode("overwrite").save(vacuum_demo_path)

tbl_delta = DeltaTable.forPath(spark, vacuum_demo_path)

# 2. Bulk UPDATE (Version 1) - Modifies 15,000 rows -> creates new Parquet files, marks old ones tombstoned
print("2. Executing bulk UPDATE on 15,000 records (Version 1)...")
tbl_delta.update(
    condition=col("PolicyID") <= 15000,
    set={"Status": lit("UNDER_REVIEW"), "RiskScore": col("RiskScore") + 10}
)

# 3. Bulk DELETE (Version 2) - Deletes 10,000 rows -> marks more files tombstoned
print("3. Executing bulk DELETE on 10,000 records (Version 2)...")
tbl_delta.delete(condition=col("PolicyID") > (vacuum_rows - 10000))

# Inspect files on storage before vacuum
files_before = [f for f in dbutils.fs.ls(vacuum_demo_path) if f.name.endswith(".parquet")]
size_before_mb = sum([f.size for f in files_before]) / (1024 * 1024)
print(f"\nStorage State BEFORE VACUUM:")
print(f"  • Total Parquet Files : {len(files_before)} files")
print(f"  • Total Storage Size  : {size_before_mb:.2f} MB (Includes both active data + dead tombstoned files!)")

# 4. Disable retention safety check for practice (TEST ONLY)
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")

# 5. Run VACUUM DRY RUN
print("\n--- Running VACUUM DRY RUN (Audits tombstoned files to be erased) ---")
tbl_delta.vacuum(0.0).show(10, truncate=False)

# 6. Run actual VACUUM to permanently purge dead files
print("Executing VACUUM RETAIN 0 HOURS to physically delete dead files...")
spark.sql(f"VACUUM delta.`{vacuum_demo_path}` RETAIN 0 HOURS")

# Inspect files after vacuum
files_after = [f for f in dbutils.fs.ls(vacuum_demo_path) if f.name.endswith(".parquet")]
size_after_mb = sum([f.size for f in files_after]) / (1024 * 1024)
print(f"\nStorage State AFTER VACUUM:")
print(f"  • Total Parquet Files : {len(files_after)} files (Dead files removed: {len(files_before) - len(files_after)})")
print(f"  • Total Storage Size  : {size_after_mb:.2f} MB (Reclaimed: {size_before_mb - size_after_mb:.2f} MB)")

# Re-enable safety check
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "true")

print("\nCritical Interview Note: Once VACUUM deletes files, Time Travel queries before that version will FAIL")
print("with 'FileNotFoundException' because the underlying Parquet files have been permanently erased!")

In [ ]:
# Task 6: Delta Liquid Clustering (CLUSTER BY) with 150,000+ Rows
# Replaces Hive-style partitioning and Z-Ordering. Multi-dimensional clustering on 150,000 rows.

cluster_rows = int(150000 * scale_multiplier)
cluster_demo_path = f"{active_base_path}/liquid_clustering_large_demo"

print(f"1. Creating Liquid Clustered Table (CLUSTER BY Region, CustomerID) at: {cluster_demo_path}...")
spark.sql(f"""
CREATE OR REPLACE TABLE delta.`{cluster_demo_path}` (
    PolicyID INT,
    CustomerID INT,
    Region STRING,
    PolicyType STRING,
    EffectiveDate DATE,
    Premium DOUBLE
)
USING DELTA
CLUSTER BY (Region, CustomerID)
""")

# Ingest 150,000 rows
print(f"2. Ingesting {cluster_rows:,} rows into clustered table...")
df_cluster = spark.range(1, cluster_rows + 1) \
    .withColumnRenamed("id", "PolicyID") \
    .withColumn("CustomerID", (col("PolicyID") % 15000) + 1) \
    .withColumn("Region", when(col("PolicyID") % 4 == 0, "West").when(col("PolicyID") % 4 == 1, "East").when(col("PolicyID") % 4 == 2, "North").otherwise("South")) \
    .withColumn("PolicyType", when(col("PolicyID") % 3 == 0, "Commercial").when(col("PolicyID") % 3 == 1, "Home").otherwise("Auto")) \
    .withColumn("EffectiveDate", date_add(to_date(lit("2024-01-01")), (col("PolicyID") % 730).cast("int"))) \
    .withColumn("Premium", (rand() * 3000 + 300).cast("double"))

df_cluster.write.format("delta").mode("append").save(cluster_demo_path)

# Run OPTIMIZE
print("3. Running OPTIMIZE to compact and cluster data layout...")
opt_res = spark.sql(f"OPTIMIZE delta.`{cluster_demo_path}`")
opt_res.select("metrics.numFilesAdded", "metrics.numFilesRemoved", "metrics.dataSizeAdded").show(truncate=False)

# Inspect clustering metadata
print("4. Inspecting Table Clustering Metadata (DESCRIBE DETAIL)...")
detail_df = spark.sql(f"DESCRIBE DETAIL delta.`{cluster_demo_path}`")
detail_df.select("format", "clusteringColumns", "numFiles", "sizeInBytes").show(truncate=False)

print("Interview Tip: Liquid Clustering completely eliminates the need to run OPTIMIZE ... ZORDER BY.")
print("To change clustering keys on the fly, simply execute: ALTER TABLE ... CLUSTER BY (NewColumn1, NewColumn2)!")

In [ ]:
# Task 7: Views Architecture with 50,000 Rows (Local Temp vs Global Temp vs Semantic Gold View)
# Demonstrates view scopes, lifecycles, and KPI aggregations on 50,000 records.

view_rows = int(50000 * scale_multiplier)
print(f"Generating {view_rows:,} customer policy records for views demo...")

df_cust_view = spark.range(1, view_rows + 1) \
    .withColumnRenamed("id", "CustomerID") \
    .withColumn("Segment", when(col("CustomerID") % 3 == 0, "Corporate").when(col("CustomerID") % 3 == 1, "SME").otherwise("Retail")) \
    .withColumn("Region", when(col("CustomerID") % 4 == 0, "North").when(col("CustomerID") % 4 == 1, "South").when(col("CustomerID") % 4 == 2, "East").otherwise("West")) \
    .withColumn("AnnualPremium", (rand() * 5000 + 500).cast("double")) \
    .withColumn("ClaimAmount", (when(col("CustomerID") % 4 == 0, rand() * 2000).otherwise(0.0)).cast("double")) \
    .withColumn("IsActive", when(col("CustomerID") % 8 == 0, False).otherwise(True))

# 1. Local Temp View
df_cust_view.createOrReplaceTempView("vw_local_customers_large")
print("1. Local Temp View created: 'vw_local_customers_large' (Current Session)")
spark.sql("SELECT Segment, count(*) as Customers, round(sum(AnnualPremium), 2) as TotalPremium FROM vw_local_customers_large GROUP BY Segment").show()

# 2. Global Temp View
df_cust_view.createOrReplaceGlobalTempView("vw_global_customers_large")
print("2. Global Temp View created in 'global_temp.vw_global_customers_large' (Cluster-wide)")
spark.sql("SELECT Region, count(*) as ActiveCount FROM global_temp.vw_global_customers_large WHERE IsActive = true GROUP BY Region").show()

# 3. Semantic Gold View
print("3. Demonstrating Semantic Business KPI View (Executive BI Metric Layer):")
semantic_view_sql = """
SELECT 
    Segment,
    Region,
    COUNT(CustomerID) AS TotalCustomers,
    ROUND(SUM(AnnualPremium), 2) AS TotalWrittenPremium,
    ROUND(SUM(ClaimAmount), 2) AS TotalIncurredClaims,
    ROUND((SUM(ClaimAmount) / SUM(AnnualPremium)) * 100, 2) AS LossRatioPercentage,
    ROUND(AVG(AnnualPremium), 2) AS AveragePremiumPerCustomer
FROM vw_local_customers_large
GROUP BY Segment, Region
ORDER BY Segment, Region
"""
df_semantic = spark.sql(semantic_view_sql)
df_semantic.show(12, truncate=False)

print("\n" + "="*80)
print(f"{'Feature':<22} | {'Local Temp View':<18} | {'Global Temp View':<18} | {'Semantic Gold View':<20}")
print("-"*80)
print(f"{'Scope':<22} | {'Current Session':<18} | {'Cluster-wide':<18} | {'Enterprise-wide (UC)':<20}")
print(f"{'Catalog Storage':<22} | {'In-Memory Only':<18} | {'global_temp DB':<18} | {'Metastore / Unity Catalog':<20}")
print(f"{'BI Tool Accessible?':<22} | {'No':<18} | {'No':<18} | {'Yes (PowerBI, Tableau)':<20}")
print(f"{'Primary Use Case':<22} | {'Intermediate wrangle':<18} | {'Notebook sharing':<18} | {'Single Source of Truth':<20}")
print("="*80)

In [ ]:
# Task 8: Delta Live Tables (DLT) Declarative Medallion Pipelines
# Concept:
# Delta Live Tables is a declarative ETL framework for building reliable data pipelines on Databricks.
# Key Features:
# 1. Declarative: You define 'what' tables to build, DLT builds the DAG and manages infrastructure.
# 2. Built-in Data Quality (Expectations): @dlt.expect, @dlt.expect_or_drop, @dlt.expect_or_fail.
# 3. Automatic Medallion Flow: Streaming and Batch unified.

dlt_pipeline_sample_code = '''
import dlt
from pyspark.sql.functions import col, current_timestamp, initcap, trim, lower, count, sum, avg

# --- 1. BRONZE LAYER: Raw High-Volume Data Streaming Ingestion ---
@dlt.table(
    name="bronze_policies_stream",
    comment="Raw ingested high-volume policies from Landing ADLS Gen2 container",
    table_properties={"quality": "bronze"}
)
def bronze_policies_stream():
    return (
        spark.readStream.format("cloudFiles")
            .option("cloudFiles.format", "parquet")
            .load("abfss://landing@azureprojectdevadls01.dfs.core.windows.net/policies/")
    )

# --- 2. SILVER LAYER: Data Cleansing & Data Quality Expectations ---
@dlt.table(
    name="silver_policies_cleaned",
    comment="Cleansed policies with strict data quality constraints applied",
    table_properties={"quality": "silver"}
)
@dlt.expect_or_fail("valid_policy_id", "PolicyID IS NOT NULL")
@dlt.expect_or_drop("valid_premium_amount", "Premium > 0")
@dlt.expect("warn_high_deductible", "Deductible <= 10000")  # Track metric in UI without dropping
def silver_policies_cleaned():
    return (
        dlt.read_stream("bronze_policies_stream")
            .withColumn("PolicyType", initcap(trim(col("PolicyType"))))
            .withColumn("CustomerEmail", lower(trim(col("CustomerEmail"))))
            .withColumn("SilverIngestedAt", current_timestamp())
    )

# --- 3. GOLD LAYER: Aggregated Business Semantic Reporting Table ---
@dlt.table(
    name="gold_policy_performance_kpi",
    comment="Aggregated policy metrics per State and Type for Executive BI dashboards",
    table_properties={"quality": "gold"}
)
def gold_policy_performance_kpi():
    return (
        dlt.read("silver_policies_cleaned")
            .groupBy("State", "PolicyType")
            .agg(
                count("PolicyID").alias("ActivePolicyCount"),
                sum("Premium").alias("TotalPremiumVolume"),
                avg("Premium").alias("AveragePremium")
            )
    )
'''

print("=== Delta Live Tables (DLT) Declarative Pipeline Architecture ===")
print(dlt_pipeline_sample_code)

print("\n--- DLT Expectation Action Modes for Interviews ---")
print("1. @dlt.expect('rule', 'expr')         -> Retains invalid record, logs metric failure to Event Log UI.")
print("2. @dlt.expect_or_drop('rule', 'expr') -> Drops dirty records automatically from downstream table.")
print("3. @dlt.expect_or_fail('rule', 'expr') -> Halts pipeline execution immediately on any critical violation!")

In [ ]:
# Task 9: SCD Type 1 & SCD Type 2 at Enterprise Scale (50,000 Base Rows + 10,000 Incoming CDC)
# Pure PySpark Code - NO SQL strings used!

base_dim_rows     = int(50000 * scale_multiplier)
incoming_cdc_rows = int(10000 * scale_multiplier)

scd1_path = f"{active_base_path}/scd_type1_large_demo"
scd2_path = f"{active_base_path}/scd_type2_large_demo"

# =========================================================================
# PART A: SCD TYPE 1 (In-Place Overwrite on 50,000 Rows)
# =========================================================================
print("\n" + "="*75)
print(f"  PART A: SCD TYPE 1 (OVERWRITE IN-PLACE) - {base_dim_rows:,} BASE + {incoming_cdc_rows:,} CDC")
print("="*75)

# Generate 50,000 base customer records
df_scd1_base = spark.range(1, base_dim_rows + 1) \
    .withColumnRenamed("id", "CustomerID") \
    .withColumn("FullName", concat_ws(" ", lit("Customer_"), col("CustomerID").cast("string"))) \
    .withColumn("City", when(col("CustomerID") % 3 == 0, "Dallas").when(col("CustomerID") % 3 == 1, "Miami").otherwise("Chicago")) \
    .withColumn("State", when(col("CustomerID") % 3 == 0, "TX").when(col("CustomerID") % 3 == 1, "FL").otherwise("IL")) \
    .withColumn("LastModified", to_date(lit("2026-01-01")))

df_scd1_base.write.format("delta").mode("overwrite").save(scd1_path)
print(f"Initial SCD Type 1 Table Ingested ({spark.read.format('delta').load(scd1_path).count():,} rows)")

# Generate 10,000 incoming CDC updates (7,000 updates to existing IDs + 3,000 brand-new customer IDs)
df_scd1_incoming = spark.range(base_dim_rows - 7000 + 1, base_dim_rows + 3000 + 1) \
    .withColumnRenamed("id", "CustomerID") \
    .withColumn("FullName", concat_ws(" ", lit("Customer_"), col("CustomerID").cast("string"))) \
    .withColumn("City", lit("Austin")) \
    .withColumn("State", lit("TX")) \
    .withColumn("LastModified", to_date(lit("2026-03-01")))

# Execute Pure PySpark MERGE
delta_scd1 = DeltaTable.forPath(spark, scd1_path)
delta_scd1.alias("tgt").merge(
    source=df_scd1_incoming.alias("src"),
    condition="tgt.CustomerID = src.CustomerID"
).whenMatchedUpdate(set={
    "City": col("src.City"),
    "State": col("src.State"),
    "LastModified": col("src.LastModified")
}).whenNotMatchedInsert(values={
    "CustomerID": col("src.CustomerID"),
    "FullName": col("src.FullName"),
    "City": col("src.City"),
    "State": col("src.State"),
    "LastModified": col("src.LastModified")
}).execute()

df_scd1_result = spark.read.format("delta").load(scd1_path)
print(f"Target Table after SCD Type 1 MERGE: Total Rows = {df_scd1_result.count():,} (Expected: {base_dim_rows + 3000:,})")
print("Sample rows where City was updated in-place to Austin:")
df_scd1_result.filter(col("City") == "Austin").show(5, truncate=False)

# =========================================================================
# PART B: SCD TYPE 2 (Full Historical Tracking with Dates & IsCurrent)
# =========================================================================
print("\n" + "="*75)
print(f"  PART B: SCD TYPE 2 (HISTORICAL TRACKING) - {base_dim_rows:,} BASE + {incoming_cdc_rows:,} CDC")
print("="*75)

# Schema: CustomerSK, CustomerID, FullName, City, State, EffectiveStartDate, EffectiveEndDate, IsCurrent
df_scd2_base = spark.range(1, base_dim_rows + 1) \
    .withColumnRenamed("id", "CustomerSK") \
    .withColumn("CustomerID", col("CustomerSK")) \
    .withColumn("FullName", concat_ws(" ", lit("Customer_"), col("CustomerID").cast("string"))) \
    .withColumn("City", when(col("CustomerID") % 3 == 0, "Dallas").when(col("CustomerID") % 3 == 1, "Miami").otherwise("Chicago")) \
    .withColumn("State", when(col("CustomerID") % 3 == 0, "TX").when(col("CustomerID") % 3 == 1, "FL").otherwise("IL")) \
    .withColumn("EffectiveStartDate", to_date(lit("2025-01-01"))) \
    .withColumn("EffectiveEndDate", to_date(lit("9999-12-31"))) \
    .withColumn("IsCurrent", lit(True))

df_scd2_base.write.format("delta").mode("overwrite").save(scd2_path)
print(f"Initial SCD Type 2 Table Created ({spark.read.format('delta').load(scd2_path).count():,} active records)")

# Incoming CDC updates (7,000 existing customers move to Denver + 3,000 new customers)
df_scd2_incoming = spark.range(base_dim_rows - 7000 + 1, base_dim_rows + 3000 + 1) \
    .withColumnRenamed("id", "CustomerID") \
    .withColumn("FullName", concat_ws(" ", lit("Customer_"), col("CustomerID").cast("string"))) \
    .withColumn("City", lit("Denver")) \
    .withColumn("State", lit("CO"))

today_date = to_date(lit("2026-03-15"))
delta_scd2 = DeltaTable.forPath(spark, scd2_path)
active_target_df = spark.read.format("delta").load(scd2_path).filter(col("IsCurrent") == True)

# Step 1: Detect changed records (where active target City != incoming City)
df_joined = df_scd2_incoming.alias("inc").join(
    active_target_df.alias("tgt"),
    on="CustomerID",
    how="left"
)

rows_to_expire = df_joined.filter(
    col("tgt.CustomerID").isNotNull() & (col("inc.City") != col("tgt.City"))
).select("tgt.CustomerSK")

# Step 2: Expire historical records in target Delta table (Close out old version)
delta_scd2.alias("tgt").merge(
    source=rows_to_expire.alias("exp"),
    condition="tgt.CustomerSK = exp.CustomerSK"
).whenMatchedUpdate(set={
    "EffectiveEndDate": today_date,
    "IsCurrent": lit(False)
}).execute()

# Step 3: Insert new active versions (both new customers + new active row for updated customers)
max_sk = spark.read.format("delta").load(scd2_path).agg({"CustomerSK": "max"}).collect()[0][0] or 0
w_sk = Window.orderBy("CustomerID")

rows_to_insert = df_joined.filter(
    col("tgt.CustomerID").isNull() | (col("inc.City") != col("tgt.City"))
).select(
    "inc.CustomerID",
    "inc.FullName",
    "inc.City",
    "inc.State"
).withColumn("CustomerSK", row_number().over(w_sk) + lit(max_sk)) \
 .withColumn("EffectiveStartDate", today_date) \
 .withColumn("EffectiveEndDate", to_date(lit("9999-12-31"))) \
 .withColumn("IsCurrent", lit(True))

# Append new active rows
rows_to_insert.select(
    "CustomerSK", "CustomerID", "FullName", "City", "State", "EffectiveStartDate", "EffectiveEndDate", "IsCurrent"
).write.format("delta").mode("append").save(scd2_path)

# Verification
df_scd2_final = spark.read.format("delta").load(scd2_path)
total_count   = df_scd2_final.count()
active_count  = df_scd2_final.filter(col("IsCurrent") == True).count()
history_count = df_scd2_final.filter(col("IsCurrent") == False).count()

print("\n--- SCD Type 2 Verification Report ---")
print(f"Total Rows in Table       : {total_count:,} (50,000 initial + 7,000 new versions + 3,000 new customers)")
print(f"Active Records (True)     : {active_count:,} (Current state of all customers)")
print(f"Historical Records (False): {history_count:,} (Preserved audit trail of past addresses)")
print("\nSample Customer showing BOTH Historical Row and Current Active Row:")
df_scd2_final.filter(col("CustomerID") == (base_dim_rows - 5000)).show(truncate=False)

In [ ]:
# Task 10: How CDC Works & Delta Change Data Feed (CDF) at Scale (20,000+ Records)
# Generates 20,000 initial policies, updates 5,000, deletes 2,000, inserts 3,000,
# and reads the complete Change Data Feed audit stream.

cdc_rows = int(20000 * scale_multiplier)
cdc_demo_path = f"{active_base_path}/cdc_cdf_large_demo"

# 1. Create table with enableChangeDataFeed
print(f"1. Creating Delta Table with Change Data Feed enabled ({cdc_rows:,} rows)...")
df_cdc_init = spark.range(1, cdc_rows + 1) \
    .withColumnRenamed("id", "PolicyID") \
    .withColumn("PolicyName", concat_ws("-", lit("PLAN"), (col("PolicyID") + 50000).cast("string"))) \
    .withColumn("Premium", (rand() * 2000 + 200).cast("double"))

df_cdc_init.write \
    .format("delta") \
    .mode("overwrite") \
    .option("delta.enableChangeDataFeed", "true") \
    .save(cdc_demo_path)

tbl_cdc = DeltaTable.forPath(spark, cdc_demo_path)

# 2. Bulk UPDATE (5,000 policies get 15% rate increase)
print("2. Performing bulk UPDATE on 5,000 policies (Version 1)...")
tbl_cdc.update(
    condition=col("PolicyID") <= 5000,
    set={"Premium": col("Premium") * 1.15}
)

# 3. Bulk DELETE (2,000 expired policies cancelled)
print("3. Performing bulk DELETE on 2,000 policies (Version 2)...")
tbl_cdc.delete(condition=(col("PolicyID") > 5000) & (col("PolicyID") <= 7000))

# 4. Bulk INSERT (3,000 brand new policies added)
print("4. Performing bulk INSERT of 3,000 new policies (Version 3)...")
spark.range(cdc_rows + 1, cdc_rows + 3001) \
    .withColumnRenamed("id", "PolicyID") \
    .withColumn("PolicyName", concat_ws("-", lit("PLAN"), (col("PolicyID") + 50000).cast("string"))) \
    .withColumn("Premium", (rand() * 2500 + 350).cast("double")) \
    .write.format("delta").mode("append").save(cdc_demo_path)

# 5. Read Change Data Feed across all versions
print("\n" + "="*85)
print("  DELTA CHANGE DATA FEED (CDF) SYSTEM AUDIT STREAM")
print("="*85)

df_cdf_changes = spark.read.format("delta") \
    .option("readChangeFeed", "true") \
    .option("startingVersion", 0) \
    .load(cdc_demo_path)

print(f"Total Change Records Captured: {df_cdf_changes.count():,}")
print("\nChange Types Breakdown in Commit Stream:")
df_cdf_changes.groupBy("_change_type").count().show()

print("Sample CDF Records (Showing pre-image and post-image for an updated policy):")
df_cdf_changes.filter(col("PolicyID") == 100).select(
    "PolicyID", "PolicyName", "Premium", "_change_type", "_commit_version", "_commit_timestamp"
).show(truncate=False)

print("Key Interview Explanations for _change_type values:")
print("  • 'insert'            : Newly inserted records (Version 0 and Version 3).")
print("  • 'update_preimage'   : Row values BEFORE the update occurred (Old Premium = 1200.0).")
print("  • 'update_postimage'  : Row values AFTER the update occurred (New Premium = 1550.0).")
print("  • 'delete'            : Records that were deleted (Policy 2).")
print("\nDownstream consumers (e.g. Silver to Gold streaming pipelines) read ONLY these change feeds")
print("to process incremental updates without doing expensive full table scans!")

In [ ]:
# Task 11: Optional Practice Environment Cleanup
# Set cleanup_enabled = True when you wish to delete the temporary demo directories created in this notebook.

cleanup_enabled = False  # Toggle to True to purge practice folders

if cleanup_enabled:
    demo_paths = [
        partition_demo_path,
        benchmark_base,
        vacuum_demo_path,
        cluster_demo_path,
        scd1_path,
        scd2_path,
        cdc_demo_path
    ]
    print("Cleaning up temporary practice directories...")
    for p in demo_paths:
        try:
            dbutils.fs.rm(p, recurse=True)
            print(f"  [Deleted]: {p}")
        except Exception as e_del:
            print(f"  [Skip]: {p} ({e_del})")
    print("Cleanup completed successfully.")
else:
    print("Cleanup is disabled (cleanup_enabled = False).")
    print(f"All demo tables and storage paths remain preserved at: {active_base_path}")